In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit
import seaborn as sns
import bayesflow as bf

import sys
import os
sys.path.append("../")

from src.generative_models import sample_pt_model, sample_mvl_model, get_choice, get_pt_utility
from src.context import get_context
from src.priors import sample_pt_prior, sample_mvl_prior


2025-02-05 09:40:58.371981: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-05 09:40:58.372035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-05 09:40:58.373638: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-05 09:40:58.385655: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-05 09:41:00.412243: W tensorflow/compiler/tf2

In [2]:
context_gen = bf.simulation.ContextGenerator(
    batchable_context_fun=get_context
)

## PT Model

In [3]:
param_names = (r'$\lambda$', r'$\alpha$', r'$\tau$')

PT_PRIOR_MEANS = np.array([1.7, 0.7, 6.])
PT_PRIOR_STDS = np.array([0.7, 0.3, 6.])

In [4]:
prior = bf.simulation.Prior(
    batch_prior_fun=sample_pt_prior,
    param_names=param_names
)

In [5]:
simulator = bf.simulation.Simulator(
    simulator_fun=sample_pt_model,
    context_generator=context_gen
)

model = bf.simulation.GenerativeModel(
    prior=prior,
    simulator=simulator,
    name="pt_model"
)

INFO:root:Performing 2 pilot runs with the pt_model model...
INFO:root:Shape of parameter batch after 2 pilot simulations: (batch_size = 2, 3)
INFO:root:Shape of simulation batch after 2 pilot simulations: (batch_size = 2, 160)
INFO:root:No optional prior non-batchable context provided.
INFO:root:No optional prior batchable context provided.
INFO:root:No optional simulation non-batchable context provided.
INFO:root:Could not determine shape of simulation batchable context. Type appears to be non-array: <class 'list'>,                                    so make sure your input configurator takes cares of that!


In [6]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

2025-02-05 09:41:11.203231: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11526 MB memory:  -> device: 0, name: NVIDIA TITAN Xp, pci bus id: 0000:05:00.0, compute capability: 6.1


In [7]:
def configurator(forward_dict):
    out_dict = {}
    data = forward_dict["sim_data"][:, :, None]
    context = np.array(forward_dict["sim_batchable_context"]) / 200
    out_dict["summary_conditions"] = np.c_[data, context].astype(np.float32)
    params = forward_dict["prior_draws"].astype(np.float32)
    out_dict["parameters"] = (params - PT_PRIOR_MEANS) / PT_PRIOR_STDS
    return out_dict

In [10]:
amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

trainer = bf.trainers.Trainer(
    generative_model=model, 
    amortizer=amortizer, 
    configurator=configurator, 
    checkpoint_path=f"../checkpoints/{model.name}",
    max_to_keep=1
)

INFO:root:Initialized empty loss history.
INFO:root:Initialized networks from scratch.
INFO:root:Performing a consistency check with provided components...
INFO:root:Done.


In [ ]:
history = trainer.train_online(100, 1000, 128)

Training epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]2025-02-05 09:41:49.578602: I external/local_xla/xla/service/service.cc:168] XLA service 0x1464b64911e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-02-05 09:41:49.578633: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA TITAN Xp, Compute Capability 6.1
2025-02-05 09:41:49.585121: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-02-05 09:41:49.606600: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1738744909.700437  754033 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Training epoch 58:   8%|▊         | 81/1000 [00:10<02:05,  7.31it/s, Epoch: 58, Iter: 81,Loss: -2.363,Avg.Loss: -2.272,LR: 1.95E-04]

## MVL model

In [3]:
param_names = (r'$\b_{var}$', r'$\b_{loss}$', r'$\tau$')

MVL_PRIOR_MEANS = np.array([0.3, 0.3, 0.5])
MVL_PRIOR_STDS = np.array([0.26, 0.26, 0.5])

In [4]:
prior = bf.simulation.Prior(
    batch_prior_fun=sample_mvl_prior,
    param_names=param_names
)

In [ ]:
simulator = bf.simulation.Simulator(
    simulator_fun=sample_mvl_model,
    context_generator=context_gen
)

model = bf.simulation.GenerativeModel(
    prior=prior,
    simulator=simulator,
    name="mvl_model"
)

In [ ]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

In [7]:
def configurator(forward_dict):
    out_dict = {}
    data = forward_dict["sim_data"][:, :, None]
    context = np.array(forward_dict["sim_batchable_context"]) / 200
    out_dict["summary_conditions"] = np.c_[data, context].astype(np.float32)
    params = forward_dict["prior_draws"]
    out_dict["parameters"] = ((params - MVL_PRIOR_MEANS) / MVL_PRIOR_STDS).astype(np.float32)
    return out_dict

In [ ]:
amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

trainer = bf.trainers.Trainer(
    generative_model=model, 
    amortizer=amortizer, 
    configurator=configurator, 
    checkpoint_path=f"../checkpoints/{model.name}",
    max_to_keep=1
)

In [ ]:
history = trainer.train_online(100, 1000, 128)